In [1]:
import sys
import os
import re
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

sys.path.append("..")
from data.base import get_connection

print("Dependencies loaded successfully for Notebook 07c.")

# 1. Define Road Geometry Classification Logic
MAJOR_ARTERIALS = [
    'BROADWAY', 'BOWERY', 'QUEENS BLVD', 'EASTERN PKWY', 'OCEAN PKWY',
    'ATLANTIC AVE', 'FLATBUSH AVE', 'GRAND CONCOURSE', 'NORTHERN BLVD',
    'WOODHAVEN BLVD', 'LINDEN BLVD', 'HYLAN BLVD', 'CROSS BRONX',
    'CONDUIT AVE', 'MEEKER AVE', 'BRUCKNER BLVD', 'STATEN ISLAND EXPY'
]

def classify_road_geometry(query_str):
    if not isinstance(query_str, str):
        return 'Standard 2-Way Grid'
    
    clean = query_str.replace(', NEW YORK, NY', '').strip().upper()
    
    # Check for multi-way/split delimiters
    and_count = len(re.findall(r'\bAND\b', clean))
    dash_count = clean.count(' - ')
    slash_count = clean.count('/')
    
    # 5-way/6-way intersections, complex junctions, or triple named crossings
    if and_count >= 2 or dash_count >= 1 or slash_count >= 1:
        return 'Multi-Road / Complex Node'
    
    # Major multi-lane diagonal or express corridors
    if any(arterial in clean for arterial in MAJOR_ARTERIALS):
        return 'Arterial / Diagonal Corridor'
    
    return 'Standard 2-Way Grid'

# Test classification logic
test_samples = [
    "5TH AVE AND 42ND ST, NEW YORK, NY",
    "BROADWAY AND 5TH AVE AND 23RD ST, NEW YORK, NY",
    "QUEENS BLVD AND 55TH AVE, NEW YORK, NY"
]
for s in test_samples:
    print(f"Sample: {s} -> {classify_road_geometry(s)}")

Dependencies loaded successfully for Notebook 07c.
Sample: 5TH AVE AND 42ND ST, NEW YORK, NY -> Standard 2-Way Grid
Sample: BROADWAY AND 5TH AVE AND 23RD ST, NEW YORK, NY -> Multi-Road / Complex Node
Sample: QUEENS BLVD AND 55TH AVE, NEW YORK, NY -> Arterial / Diagonal Corridor


In [2]:
# Load permanent camera coordinates master file
cams_df = pd.read_csv('../data/current_active_permanent_cameras_2026.csv')

# Apply classification
cams_df['road_geometry'] = cams_df['search_query'].apply(classify_road_geometry)

print("=== Permanent Camera Distribution Across Road Geometries ===")
geo_dist = cams_df.groupby(['road_geometry', 'camera_type']).size().unstack(fill_value=0)
geo_dist['Total'] = geo_dist.sum(axis=1)
geo_dist['Share_%'] = (geo_dist['Total'] / len(cams_df) * 100).round(2)
print(geo_dist.to_string())

# Register in DuckDB session for downstream joins
con = get_connection()
con.register('cams_classified', cams_df[['search_query', 'road_geometry', 'camera_type']])

=== Permanent Camera Distribution Across Road Geometries ===
camera_type                   Red Light  Speed  Total  Share_%
road_geometry                                                 
Arterial / Diagonal Corridor        137    250    387    14.68
Multi-Road / Complex Node            10     45     55     2.09
Standard 2-Way Grid                 444   1750   2194    83.23


In [3]:
query = """
    WITH cleaned_tickets AS (
        SELECT 
            CAST(t.issue_date AS DATE) AS date,
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            TRIM(UPPER(COALESCE(t.street_name, '') || ' ' || COALESCE(t.intersecting_street, ''))),
                            '\\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\\b', '', 'g'
                        ),
                        '\\(.*?\\)', '', 'g'
                    ),
                    '@', ' AND ', 'g'
                ),
                '\\s+', ' ', 'g'
            ) || ', NEW YORK, NY' AS search_query
        FROM tickets_all t
        WHERE CAST(t.violation_code AS INTEGER) IN (7, 36)
          AND t.issue_date >= '2016-01-01'
          AND t.street_name IS NOT NULL 
          AND t.intersecting_street IS NOT NULL
    ),
    weekly_tickets AS (
        SELECT 
            DATE_TRUNC('week', t.date) AS week_date,
            t.search_query,
            COUNT(*) AS tickets_issued
        FROM cleaned_tickets t
        INNER JOIN cams_classified c ON t.search_query = c.search_query
        GROUP BY 1, 2
    ),
    weekly_crashes AS (
        SELECT 
            DATE_TRUNC('week', CAST(crash_date AS DATE)) AS week_date,
            search_query,
            COUNT(*) AS crash_count,
            SUM(COALESCE(number_of_persons_injured, 0)) AS injured_count,
            SUM(COALESCE(number_of_pedestrians_injured, 0)) AS ped_injured_count,
            SUM(COALESCE(number_of_cyclist_injured, 0)) AS cyclist_injured_count
        FROM camera_crash_buffers
        GROUP BY 1, 2
    ),
    weekly_backbone AS (
        SELECT 
            COALESCE(t.week_date, c.week_date) AS week_date,
            COALESCE(t.search_query, c.search_query) AS search_query,
            COALESCE(t.tickets_issued, 0) AS tickets_issued,
            COALESCE(c.crash_count, 0) AS crash_count,
            COALESCE(c.injured_count, 0) AS injured_count,
            COALESCE(c.ped_injured_count, 0) AS ped_injured_count,
            COALESCE(c.cyclist_injured_count, 0) AS cyclist_injured_count
        FROM weekly_tickets t
        FULL OUTER JOIN weekly_crashes c 
          ON t.week_date = c.week_date AND t.search_query = c.search_query
    )
    SELECT 
        b.*,
        c.road_geometry,
        c.camera_type,
        a.inferred_activation_date,
        CASE 
            WHEN a.inferred_activation_date IS NOT NULL AND b.week_date >= a.inferred_activation_date 
            THEN 1 
            ELSE 0 
        END AS is_camera_active
    FROM weekly_backbone b
    JOIN cams_classified c ON b.search_query = c.search_query
    LEFT JOIN camera_activation_dates a ON b.search_query = a.search_query
    ORDER BY week_date, search_query
"""

try:
    print("Extracting weekly geometry panel from DuckDB...")
    geo_panel = con.sql(query).df()
    print(f"Extracted {len(geo_panel):,} weekly camera-intersection observations.")
finally:
    con.close()

geo_panel['week_date'] = pd.to_datetime(geo_panel['week_date'])
geo_panel['month'] = geo_panel['week_date'].dt.month.astype(str)
geo_panel['year'] = geo_panel['week_date'].dt.year.astype(str)
geo_panel['vru_injured_count'] = geo_panel['ped_injured_count'] + geo_panel['cyclist_injured_count']

geo_panel.head()

Extracting weekly geometry panel from DuckDB...
Extracted 243,391 weekly camera-intersection observations.


,week_date,search_query,tickets_issued,crash_count,injured_count,ped_injured_count,cyclist_injured_count,road_geometry,camera_type,inferred_activation_date,is_camera_active,month,year,vru_injured_count
0,2015-12-28,"10TH AVE AND LITTLE W 12TH ST, NEW YORK, NY",0,1,0.0,0.0,0.0,Standard 2-Way Grid,Speed,2019-11-25,0,12,2015,0.0
1,2015-12-28,"112TH ST AND JAMAIC A AVE, NEW YORK, NY",0,1,1.0,0.0,1.0,Standard 2-Way Grid,Speed,2025-02-03,0,12,2015,1.0
2,2015-12-28,"12TH AVE AND 34T H ST, NEW YORK, NY",0,1,0.0,0.0,0.0,Standard 2-Way Grid,Red Light,2016-05-30,0,12,2015,0.0
3,2015-12-28,"130TH ST AND N COND UIT AVE, NEW YORK, NY",0,1,0.0,0.0,0.0,Standard 2-Way Grid,Red Light,2026-04-27,0,12,2015,0.0
4,2015-12-28,"1ST AVE AND E 116TH ST, NEW YORK, NY",0,1,0.0,0.0,0.0,Standard 2-Way Grid,Speed,2025-08-04,0,12,2015,0.0


In [4]:
print("=== Descriptive Geometry Disparities Across Camera Sites ===")
geo_summary = geo_panel.groupby('road_geometry').agg(
    total_intersections=('search_query', 'nunique'),
    weekly_mean_crashes=('crash_count', 'mean'),
    weekly_mean_injuries=('injured_count', 'mean'),
    weekly_mean_vru_injuries=('vru_injured_count', 'mean'),
    weekly_mean_tickets=('tickets_issued', 'mean')
).round(3)

print(geo_summary.to_string())

print("\n=== Model 1: Baseline Crash Risk by Roadway Geometry (Poisson GLM) ===")
# Base risk model holding camera operational status, camera type, and time constant
model_geo_base = smf.glm(
    """
    crash_count ~ C(road_geometry, Treatment(reference='Standard 2-Way Grid')) 
                  + is_camera_active + C(camera_type) + C(month) + C(year)
    """,
    data=geo_panel,
    family=sm.families.Poisson()
).fit(cov_type='cluster', cov_kwds={'groups': geo_panel['search_query']})

print(model_geo_base.summary().tables[1])

# Calculate excess baseline crash multiplier
for geo in ['Arterial / Diagonal Corridor', 'Multi-Road / Complex Node']:
    coef = model_geo_base.params[f"C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.{geo}]"]
    pct = (np.exp(coef) - 1) * 100
    print(f"Excess Crash Rate on {geo} vs Standard Grid: {pct:+.2f}%")

=== Descriptive Geometry Disparities Across Camera Sites ===
                              total_intersections  weekly_mean_crashes  weekly_mean_injuries  weekly_mean_vru_injuries  weekly_mean_tickets
road_geometry                                                                                                                              
Arterial / Diagonal Corridor                  365                0.903                 0.409                     0.079               32.332
Multi-Road / Complex Node                      50                0.977                 0.414                     0.114                4.072
Standard 2-Way Grid                          2116                0.915                 0.410                     0.102               19.538

=== Model 1: Baseline Crash Risk by Roadway Geometry (Poisson GLM) ===
                                                                                                   coef    std err          z      P>|z|      [0.025      0.975]
------

In [5]:
print("=== Model 2: Camera Treatment Effect by Geometry (Poisson Interaction Model) ===")

# Tests whether camera deterrence is stronger or attenuated at complex vs simple intersections
model_geo_interact = smf.glm(
    """
    crash_count ~ is_camera_active * C(road_geometry, Treatment(reference='Standard 2-Way Grid')) 
                  + C(camera_type) + C(month) + C(year)
    """,
    data=geo_panel,
    family=sm.families.Poisson()
).fit(cov_type='cluster', cov_kwds={'groups': geo_panel['search_query']})

print(model_geo_interact.summary().tables[1])

# Quantify net treatment effect across each roadway category
beta_base = model_geo_interact.params['is_camera_active']
grid_effect = (np.exp(beta_base) - 1) * 100

beta_arterial = beta_base + model_geo_interact.params["is_camera_active:C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Arterial / Diagonal Corridor]"]
arterial_effect = (np.exp(beta_arterial) - 1) * 100

beta_complex = beta_base + model_geo_interact.params["is_camera_active:C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Multi-Road / Complex Node]"]
complex_effect = (np.exp(beta_complex) - 1) * 100

print(f"\nNet Camera Crash Effect on Standard 2-Way Grid:        {grid_effect:+.2f}%")
print(f"Net Camera Crash Effect on Arterial / Diagonal Corridor: {arterial_effect:+.2f}%")
print(f"Net Camera Crash Effect on Multi-Road / Complex Node:     {complex_effect:+.2f}%")

=== Model 2: Camera Treatment Effect by Geometry (Poisson Interaction Model) ===
                                                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                                                        -0.0494      0.037     -1.352      0.176      -0.121       0.022
C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Arterial / Diagonal Corridor]                      0.1095      0.033      3.315      0.001       0.045       0.174
C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Multi-Road / Complex Node]                        -0.0753      0.047     -1.592      0.111      -0.168       0.017
C(camera_type)[T.Speed]      

In [6]:
print("=== Model 3: Pedestrian and Cyclist Injury Reductions Across Geometries ===")

model_vru = smf.glm(
    """
    vru_injured_count ~ is_camera_active * C(road_geometry, Treatment(reference='Standard 2-Way Grid')) 
                        + C(camera_type) + C(month) + C(year)
    """,
    data=geo_panel,
    family=sm.families.Poisson()
).fit(cov_type='cluster', cov_kwds={'groups': geo_panel['search_query']})

print(model_vru.summary().tables[1])

# Save consolidated regression output to CSV
os.makedirs('../data/output', exist_ok=True)
summary_table = pd.DataFrame({
    'Crash_Model_Coef': model_geo_interact.params,
    'Crash_Model_Pval': model_geo_interact.pvalues,
    'VRU_Model_Coef': model_vru.params,
    'VRU_Model_Pval': model_vru.pvalues
})
summary_table.to_csv('../data/output/roadway_geometry_model_results.csv')
print("\nExported regression parameters to ../data/output/roadway_geometry_model_results.csv")

=== Model 3: Pedestrian and Cyclist Injury Reductions Across Geometries ===
                                                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                                                        -3.7193      0.374     -9.935      0.000      -4.453      -2.986
C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Arterial / Diagonal Corridor]                     -0.1649      0.057     -2.874      0.004      -0.277      -0.052
C(road_geometry, Treatment(reference='Standard 2-Way Grid'))[T.Multi-Road / Complex Node]                        -0.2236      0.169     -1.327      0.185      -0.554       0.107
C(camera_type)[T.Speed]           